# 03. Student Model Training & Regression Distillation
## Deng et al. (WWW '23 Companion) Section 3.2

This notebook demonstrates:
1. Consistency filtering on weak labels (filtering out posts with $<5/8$ agreement).
2. Preparing PyTorch regression dataset targeting continuous `teacher_soft_score` in $[-1.0, 1.0]$.
3. Fine-tuning compact modern transformer backbone (DistilBERT) using MSE Regression Loss.
4. Grid search on validation set to determine optimal classification threshold $\theta$.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.distillation.dataset import filter_by_consistency
from src.distillation.train import train_regression_student
from src.distillation.evaluate import evaluate_regression_student
from src.utils.common import DATA_PROC

In [ ]:
# Load train/val/test splits
train_df = pd.read_parquet(DATA_PROC / "train.parquet")
val_df = pd.read_parquet(DATA_PROC / "val.parquet")
test_df = pd.read_parquet(DATA_PROC / "test.parquet")

# Ensure teacher soft score exists (synthesize or use weak labels)
score_map = {"BULLISH": 1.0, "BEARISH": -1.0, "NEUTRAL": 0.0}
train_df["teacher_soft_score"] = train_df["sentiment_label"].map(score_map).fillna(0.0)
val_df["teacher_soft_score"] = val_df["sentiment_label"].map(score_map).fillna(0.0)

print(f"Training on {len(train_df)} samples, validating on {len(val_df)} samples.")

In [ ]:
# Train student regression model (1 epoch for demo)
model, tokenizer, history = train_regression_student(
    train_df=train_df.head(20),
    val_df=val_df.head(10),
    epochs=1,
    batch_size=8,
    save_name="notebook_student_regression",
)
print("Training history:", history)